# 01 · Dataset exploration

Look at the raw data before choosing preprocessing: class balance, what the
images look like, and how often MediaPipe finds a hand. The detection rate
matters directly: the landmark representation (and `require_hand`) drops images
without a detected hand, which `evaluate` reports as `coverage`.

Start the environment with `uv sync --all-extras --group notebooks` and
`uv run jupyter lab`.

In [ ]:
# Run from the repository root so params.yaml and data/ paths resolve.
import os
from pathlib import Path

if not Path("params.yaml").exists():
    os.chdir(Path.cwd().parent)
print(Path.cwd())

In [ ]:
from collections import Counter

import matplotlib.pyplot as plt

from asl.lab.config import load_config
from asl.lab.data import discover_classes, list_records

config = load_config(["params.yaml"])
classes = discover_classes(config.data.train_dir)
records = list_records(config.data.train_dir, classes)
counts = Counter(r.label for r in records)
print(f"{len(records)} training images, {len(classes)} classes")

plt.figure(figsize=(12, 3))
plt.bar(list(counts), list(counts.values()))
plt.title("Images per class")
plt.show()

In [ ]:
import random

from asl.prod.preprocessing import read_image_rgb

random.seed(0)
sample = random.sample(records, 12)
fig, axes = plt.subplots(2, 6, figsize=(14, 5))
for ax, record in zip(axes.flat, sample):
    ax.imshow(read_image_rgb(record.path))
    ax.set_title(record.label)
    ax.axis("off")
plt.show()

## Preprocessing variants

The same `Preprocessor` the API uses, under different settings. Compare what the
model would see with landmarks drawn, with a hand crop, and both.

In [ ]:
from asl.prod.preprocessing import PreprocessConfig, Preprocessor

variants = {
    "resize only": dict(draw_landmarks=False, hand_crop=False),
    "landmarks": dict(draw_landmarks=True, hand_crop=False),
    "hand crop": dict(draw_landmarks=False, hand_crop=True),
    "landmarks + crop": dict(draw_landmarks=True, hand_crop=True),
}
images = [read_image_rgb(r.path) for r in sample[:4]]
fig, axes = plt.subplots(len(images), len(variants), figsize=(10, 10))
for col, (name, overrides) in enumerate(variants.items()):
    preprocessor = Preprocessor(PreprocessConfig(img_size=96, **overrides))
    detector = preprocessor.create_detector()
    for row, image in enumerate(images):
        result = preprocessor(image, detector)
        axes[row, col].imshow(result.features)
        axes[row, col].axis("off")
        axes[0, col].set_title(name)
    if detector:
        detector.close()
plt.show()

## Hand detection rate per class (on a sample)

In [ ]:
from asl.prod.hands import HandDetector

per_class = 20
rates = {}
with HandDetector(static_image_mode=True) as detector:
    for label in classes:
        items = [r for r in records if r.label == label][:per_class]
        rates[label] = sum(detector.detect(read_image_rgb(r.path)) is not None for r in items) / max(len(items), 1)

plt.figure(figsize=(12, 3))
plt.bar(list(rates), list(rates.values()))
plt.ylim(0, 1)
plt.title("Fraction of images with a detected hand")
plt.show()